# 실습: Ollama 설치와 기본 호출

Ollama는 클라우드 API 키 없이 내 컴퓨터에서 LLM을 실행합니다.
이 노트북에서는 설치 확인, 모델 확인, 첫 호출, 응답 구조 분석을 진행합니다.


## 수업 진행 방식

- 이 실습은 Ollama 앱 또는 `ollama serve`가 실행 중이어야 합니다.
- 모델이 없으면 터미널에서 `ollama pull gemma3:1b`를 먼저 실행합니다.
- 학생 PC 사양에 따라 응답 속도와 품질이 다를 수 있습니다.
- API 비용은 없지만 모델 다운로드와 로컬 연산 시간이 필요합니다.


## 0. 준비 확인


In [1]:
import json
import os
import subprocess
from pprint import pprint

import requests
from dotenv import load_dotenv
from ollama import Client

load_dotenv()

HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")
MODEL = os.getenv("OLLAMA_MODEL", "gemma4:e2b")
client = Client(host=HOST)

print(f"Ollama host: {HOST}")
print(f"사용 모델: {MODEL}")


Ollama host: http://localhost:11434
사용 모델: gemma4:e2b


## 1. Ollama 설치 확인


In [2]:
def run_command(command: list[str]) -> None:
    try:
        completed = subprocess.run(
            command,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
            check=False,
        )
    except FileNotFoundError:
        print(f"명령을 찾을 수 없습니다: {command[0]}")
        return

    if completed.stdout:
        print(completed.stdout.strip())
    if completed.stderr:
        print(completed.stderr.strip())
    print(f"exit code: {completed.returncode}")


run_command(["ollama", "--version"])


ollama version is 0.31.1
exit code: 0


## 2. Ollama 서버 확인


In [3]:
def check_ollama_server() -> bool:
    try:
        response = requests.get(f"{HOST}/api/tags", timeout=3)
        print("status code:", response.status_code)
        response.raise_for_status()
        print("Ollama 서버가 응답합니다.")
        return True
    except requests.RequestException as exc:
        print("Ollama 서버에 연결할 수 없습니다.")
        print("해결 방법: Ollama 앱을 실행하거나 터미널에서 `ollama serve`를 실행하세요.")
        print(exc)
        return False


server_ready = check_ollama_server()


status code: 200
Ollama 서버가 응답합니다.


## 3. 설치된 모델 목록


In [4]:
def list_models() -> list[str]:
    try:
        response = requests.get(f"{HOST}/api/tags", timeout=5)
        response.raise_for_status()
    except requests.RequestException as exc:
        print("모델 목록 조회 실패:", exc)
        return []

    data = response.json()
    models = data.get("models", [])
    names = [model.get("name", "") for model in models]

    if not names:
        print("설치된 모델이 없습니다.")
    else:
        for name in names:
            print(name)
    return names


installed_models = list_models()
print(f"수업 모델 설치 여부: {MODEL in installed_models}")


hf.co/LGAI-EXAONE/EXAONE-4.0-1.2B-GGUF:Q4_K_M
python-tutor:latest
gemma4:e2b
수업 모델 설치 여부: True


## 4. 모델 다운로드


In [5]:
# 수업 중 자동 다운로드를 피하기 위해 기본값은 False입니다.
# 모델이 없다면 터미널에서 `ollama pull gemma3:1b`를 실행하거나,
# 아래 값을 True로 바꾼 뒤 셀을 실행하세요.

DOWNLOAD_MODEL = False

def progress_value(progress, key, default=None):
    if hasattr(progress, "get"):
        return progress.get(key, default)
    return getattr(progress, key, default)


if DOWNLOAD_MODEL:
    print(f"{MODEL} 다운로드를 시작합니다. 시간이 걸릴 수 있습니다.")
    for progress in client.pull(MODEL, stream=True):
        status = progress_value(progress, "status")
        digest = progress_value(progress, "digest", "")
        completed = progress_value(progress, "completed", 0)
        total = progress_value(progress, "total", 0)
        if total:
            print(f"{status} {digest[:12]} {completed}/{total}")
        else:
            print(status)
else:
    print("자동 다운로드를 건너뜁니다.")


자동 다운로드를 건너뜁니다.


## 5. 공통 도우미 함수


In [6]:
def response_text(response) -> str:
    message = getattr(response, "message", None)
    if message is not None:
        return getattr(message, "content", "") or ""
    return getattr(response, "response", "") or ""


def print_response(response) -> None:
    text = response_text(response)
    if text:
        print(text)
    else:
        print("답변 본문이 비어 있습니다. think=False 또는 num_predict 값을 확인하세요.")

def print_metrics(response) -> None:
    print("\n[응답 지표]")
    for field in [
        "model",
        "done",
        "prompt_eval_count",
        "eval_count",
        "total_duration",
        "load_duration",
        "prompt_eval_duration",
        "eval_duration",
    ]:
        value = getattr(response, field, None)
        if value is not None:
            print(f"{field}: {value}")


def call_ollama(label, request_func):
    print(f"\n=== {label} ===")
    try:
        response = request_func()
    except Exception as exc:
        print("Ollama 호출 실패:", type(exc).__name__)
        print(exc)
        print("\n확인할 것: Ollama 실행 여부, 모델 설치 여부, 모델명")
        return None

    print_response(response)
    print_metrics(response)
    return response


## 6. 첫 번째 Ollama 호출


In [7]:
response = call_ollama(
    "첫 번째 Ollama 호출",
    lambda: client.chat(
        model=MODEL,
        think=False,
        messages=[
            {
                "role": "user",
                "content": "Python을 처음 배우는 학생에게 변수의 의미를 한 문장으로 설명해줘.",
            }
        ],
        options={"num_predict": 180},
    ),
)



=== 첫 번째 Ollama 호출 ===
**변수는 데이터를 저장하는 그릇이라고 생각하면 돼요.**

[응답 지표]
model: gemma4:e2b
done: True
prompt_eval_count: 28
eval_count: 14
total_duration: 2681970600
load_duration: 1735725900
prompt_eval_duration: 269143000
eval_duration: 662511000


## 7. 응답 구조 분석


In [8]:
response = call_ollama(
    "응답 구조 분석용 호출",
    lambda: client.chat(
        model=MODEL,
        think=False,
        messages=[
            {
                "role": "user",
                "content": "딕셔너리를 초보자에게 설명하고, 짧은 예시 코드도 보여줘.",
            }
        ],
        options={"num_predict": 260},
    ),
)

if response is not None:
    print("\n[message]")
    print("role:", response.message.role)
    print("content preview:", response_text(response)[:120])

    print("\n[원본 구조 일부]")
    if hasattr(response, "model_dump"):
        dumped = response.model_dump()
    else:
        dumped = dict(response)
    pprint({key: dumped.get(key) for key in dumped.keys() if key != "message"})



=== 응답 구조 분석용 호출 ===
## 딕셔너리 (Dictionary) 완전 정복: 초보자를 위한 쉬운 설명과 예시

안녕하세요! **딕셔너리(Dictionary)**는 프로그래밍에서 데이터를 저장하는 아주 유용하고 강력한 도구입니다. 처음 들으면 조금 어려울 수 있지만, 알고 보면 정말 직관적이고 편리해요.

---

### 1. 딕셔너리란 무엇인가요? (가장 쉬운 비유)

딕셔너리는 우리가 일상생활에서 사용하는 **'사전(Dictionary)'**과 매우 비슷합니다.

**우리가 사전을 볼 때:**
* **단어 (Key):** 찾고 싶은 정보의 이름 (예: '사과')
* **뜻/정의 (Value):** 그 단어에 해당하는 정보 (예: '빨갛고, 과일이다')

마찬가지로, 딕셔너리도 **'키(Key)'**와 **'값(Value)'**의 쌍으로 데이터를 저장합니다.

> **🔑 핵심 개념:** **"이름표(Key)를 통해 원하는 정보(Value)를 아주 빠르게 찾아낼 수 있는 저장 공간"**이라고 생각하시면 됩니다.

---

### 2. 딕

[응답 지표]
model: gemma4:e2b
done: True
prompt_eval_count: 29
eval_count: 260
total_duration: 15576912500
load_duration: 1573606600
prompt_eval_duration: 260791000
eval_duration: 13735038000

[message]
role: assistant
content preview: ## 딕셔너리 (Dictionary) 완전 정복: 초보자를 위한 쉬운 설명과 예시

안녕하세요! **딕셔너리(Dictionary)**는 프로그래밍에서 데이터를 저장하는 아주 유용하고 강력한 도구입니다. 처음 들으면 

[원본 구조 일부]
{'created_at': '2026-07-03T09:42:00.5592674Z',
 'done': True,
 'done_reason': 'length',
 'eval_c

## 8. `generate` API와 `chat` API 비교


In [9]:
generate_response = call_ollama(
    "generate API",
    lambda: client.generate(
        model=MODEL,
        think=False,
        prompt="Python 함수가 왜 필요한지 3문장으로 설명해줘.",
        options={"num_predict": 180},
    ),
)

chat_response = call_ollama(
    "chat API",
    lambda: client.chat(
        model=MODEL,
        think=False,
        messages=[{"role": "user", "content": "Python 함수가 왜 필요한지 3문장으로 설명해줘."}],
        options={"num_predict": 180},
    ),
)



=== generate API ===
Python 함수는 **코드의 재사용성**을 높여줍니다. 한 번 정의된 함수를 여러 번 호출하여 동일한 작업을 효율적으로 수행할 수 있습니다. 또한, 코드를 **모듈화**하여 복잡한 프로그램을 체계적이고 이해하기 쉽게 만드는 데 필수적입니다.

[응답 지표]
model: gemma4:e2b
done: True
prompt_eval_count: 24
eval_count: 63
total_duration: 5591947100
load_duration: 1897022300
prompt_eval_duration: 253348000
eval_duration: 3429845000

=== chat API ===
Python 함수는 코드를 **재사용 가능하고**, **모듈화**하여 **관리하기 쉽게** 만들어 줍니다. 함수를 사용하면 복잡한 작업을 작은 단위로 나누어 각 부분이 독립적으로 작동하게 할 수 있어 코드의 가독성과 유지보수성을 크게 향상시킵니다. 결과적으로, 코드를 더 효율적이고 오류 없이 작성할 수 있게 해줍니다.

[응답 지표]
model: gemma4:e2b
done: True
prompt_eval_count: 24
eval_count: 83
total_duration: 6325235500
load_duration: 1636516900
prompt_eval_duration: 135849000
eval_duration: 4538221000


## 9. 시스템 메시지


In [10]:
response = call_ollama(
    "system 메시지로 역할 지정",
    lambda: client.chat(
        model=MODEL,
        think=False,
        messages=[
            {
                "role": "system",
                "content": (
                    "당신은 친절한 한국어 Python 선생님입니다. "
                    "답변은 쉬운 비유 1개, 코드 1개, 확인 질문 1개로 구성하세요."
                ),
            },
            {"role": "user", "content": "리스트가 무엇인지 설명해줘."},
        ],
        options={"num_predict": 360},
    ),
)



=== system 메시지로 역할 지정 ===
안녕하세요! 파이썬에서 '리스트(List)'에 대해 아주 쉽고 재미있게 설명해 드릴게요. 😊

### 🎈 리스트란 무엇일까요? (비유)

리스트는 **여러 개의 물건을 순서대로 담아 놓는 상자**라고 생각하시면 돼요. 이 상자 안에는 사과, 바나나, 오렌지처럼 여러 종류의 과일(데이터)들이 차곡차곡 들어있고, 그 순서가 중요하답니다.

파이썬에서는 이 리스트를 사용하여 데이터를 모아서 한 번에 관리하고, 나중에 필요할 때 원하는 물건을 쉽게 꺼내볼 수 있어요.

### 💻 코드 예시

리스트를 만드는 방법과 사용하는 예를 보여드릴게요.

```python
# 과일들을 담는 리스트를 만듭니다. (대괄호 []를 사용해요)
fruit_list = ["사과", "바나나", "오렌지"]

# 리스트의 내용을 출력해 봅니다.
print(fruit_list)

# 리스트에서 두 번째 항목(인덱스 1)을 꺼내봅니다.
print(fruit_list[1])
```

### 🤔 확인 질문

제가 설명드린 것처럼, 파이썬에서 리스트는 **순서대로 여러 데이터를 담아두는 자료구조**입니다. 혹시 이 '리스트'를 이용해서 여러분이 생각하는 다른 종류의 데이터(예: 이름 목록, 점수 목록 등)를 담아보고 싶으신가요?

[응답 지표]
model: gemma4:e2b
done: True
prompt_eval_count: 59
eval_count: 332
total_duration: 21338285600
load_duration: 1604766300
prompt_eval_duration: 489002000
eval_duration: 19232310000


## 10. 실행 중인 모델 확인


In [11]:
try:
    response = requests.get(f"{HOST}/api/ps", timeout=5)
    response.raise_for_status()
    data = response.json()
    models = data.get("models", [])
    if not models:
        print("현재 실행 중인 모델이 없습니다.")
    else:
        for model in models:
            print(model.get("name"), model.get("size_vram"))
except requests.RequestException as exc:
    print("실행 중 모델 조회 실패:", exc)


gemma4:e2b 0


## 11. 에러 처리


In [12]:
# 일부러 존재하지 않는 모델을 호출해 에러 메시지를 관찰합니다.

bad_response = call_ollama(
    "존재하지 않는 모델 테스트",
    lambda: client.chat(
        model="not-a-real-model:latest",
        think=False,
        messages=[{"role": "user", "content": "이 요청은 실패해야 정상입니다."}],
    ),
)



=== 존재하지 않는 모델 테스트 ===
Ollama 호출 실패: ResponseError
model 'not-a-real-model:latest' not found (status code: 404)

확인할 것: Ollama 실행 여부, 모델 설치 여부, 모델명


### 자주 만나는 오류

| 증상 | 확인할 것 |
| --- | --- |
| 연결 실패 | Ollama 앱 실행 여부 또는 `ollama serve` |
| model not found | `ollama pull 모델명` 실행 여부 |
| 응답이 너무 느림 | 모델 크기, RAM/VRAM, 다른 프로그램 사용량 |
| 답변 품질이 낮음 | 더 큰 모델 사용 또는 프롬프트 개선 |
